# CardionetV3

## Libraries

In [1]:
import torch
import torch.nn as nn
import torchvision
import torchaudio
import torchaudio.transforms as T

In [2]:
# === CONFIG ===
SAMPLE_RATE = 4000
IMG_SIZE = (128, 128)  # resize all spectrograms to this size Document(id='66dbad8c-cda5-4404-a5ab-d486a0a0af44', metadata={'title': 'Убийство (2016)', 'year': 2016.0, 'content_type': 'SERIAL', 'age_rating': 18.0, 'genres': ['Детектив'], 'countries': ['Великобритания']}, page_content='Новый детективный сериал производства Великобритании, возможно не с самым оригинальным названием «Убийство», выделяется среди прочих криминальных лент уникальностью стиля и сюжетом, максимально приближенным к реальности. Сериал снят методом репортажной съемки. Зритель получает сведения об убийстве от свидетелей и непосредственных участников трагического события. Детективы, расследующие дело, излагают свою точку зрения на произошедшие события, как и все остальные, развернувшись лицом к камере, то есть к нам, смотрящим в экран зрителям. По сути же, создатели сериала заставляют распутывать, додумывать и догадаться о том, что произошло, именно зрителя.Кого интересует профессия следователя – милости просим к экрану.'),

EMBED_DIM = 512
NUM_CLASSES = 3

In [6]:
# === SPECTROGRAM COMBO ===
import librosa, numpy as np, pywt, cv2

In [14]:
def get_stft(y, sr):
    D = librosa.stft(y, n_fft=512, hop_length=256)
    return np.abs(D)

def get_mel(y, sr):
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_fft=512, hop_length=256, n_mels=64)
    return librosa.power_to_db(mel)

def get_wt(y):
    coeffs, _ = pywt.cwt(y, scales=np.arange(1, 129), wavelet='morl')
    return np.abs(coeffs)

def prepare_tensor(audio, sr=SAMPLE_RATE):
    stft = get_stft(audio, sr)
    mel = get_mel(audio, sr)
    wt = get_wt(audio)

    stft = cv2.resize(stft, IMG_SIZE)
    mel = cv2.resize(mel, IMG_SIZE)
    wt = cv2.resize(wt, IMG_SIZE)

    return torch.tensor(np.stack([stft, mel, wt], axis=0), dtype=torch.float32)  # [3, H, W]






In [21]:
# === MODEL ===

class CNNTransformer(nn.Module):
    def __init__(self, n_classes=NUM_CLASSES, embedding_dim=EMBED_DIM):
        super().__init__()
        self.backbone = torchvision.models.resnet18(weights=None)
        self.backbone.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.backbone = nn.Sequential(*list(self.backbone.children())[:-2])  # remove avgpool, fc

        self.linear_proj = nn.Linear(512, embedding_dim)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embedding_dim, nhead=8, dim_feedforward=1024, dropout=0.1, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=4)

        self.cls_head = nn.Sequential(
            nn.Linear(embedding_dim, 128),
            nn.ReLU(),
            nn.Linear(128, n_classes)
        )

    def forward(self, x):  # x: [B, 3, H, W]
        x = self.backbone(x)  # [B, 512, H', W']
        B, C, H, W = x.shape
        x = x.permute(0, 2, 3, 1).reshape(B, H * W, C)  # [B, N, C]
        x = self.linear_proj(x)  # [B, N, D]
        x = self.transformer(x)  # [B, N, D]
        x = x.mean(dim=1)  # [B, D]
        return self.cls_head(x)

In [37]:
# === EXAMPLE ===
audio, sr = torchaudio.load("/home/merci/code/PycharmProjects/cardionix-pipeline/data/CardionixDataset/audio/abnormal__0a5da0db-b42b-4ddb-b280-23f9158ee505.wav")

In [17]:
x = prepare_tensor(audio.squeeze().numpy())  # [3, H, W]

In [19]:
x.shape

torch.Size([3, 128, 128])

In [22]:
model = CNNTransformer()


In [23]:
out = model(x.unsqueeze(0))  # [1, num_classes]

In [25]:
out

tensor([[ 0.0995, -0.2967, -0.1818]], grad_fn=<AddmmBackward0>)

In [41]:
audio = audio.squeeze().numpy()

stft = get_stft(audio, sr)
mel = get_mel(audio, sr)
wt = get_wt(audio)

#stft = cv2.resize(stft, IMG_SIZE)
#mel = cv2.resize(mel, IMG_SIZE)
wt = cv2.resize(wt, IMG_SIZE)

#torch.tensor(np.stack([stft, mel, wt], axis=0), dtype=torch.float32)  # [3, H, W]

In [47]:
coeffs, _ = pywt.cwt(audio, scales=np.arange(1, 129), wavelet='morl', sampling_period=12)
coeffs.shape

(128, 71332)

In [ ]:
sampling_period